# Voice → Clean Text

A low-latency "clean up my speech" pipeline, close to # how Wispr Flow works:

1. **Whisper** (fine-tuned) — turns disfluent speech directly into clean text
2. **LLM** (fine-tuned) — cleans up text further and adapts tone (email / Slack / etc.)
3. **Custom CUDA kernel** — a hand-written fused op, benchmarked for training speed
4. **Merge + push to Hugging Face Hub** — so the Gradio app can load the trained models directly from the Hub

## 1. Install packages

In [ ]:
!pip uninstall -y numpy scipy scikit-learn transformers tokenizers -q

!pip install -q --no-cache-dir \
    "numpy==2.1.3" \
    "scipy==1.14.1" \
    "scikit-learn==1.5.2" \
    "transformers==4.57.1" \
    "tokenizers>=0.22,<0.24" \
    "accelerate>=1.2" \
    "peft" \
    "bitsandbytes" \
    "evaluate" \
    "safetensors>=0.5" \
    "librosa>=0.10.2" \
    "soundfile>=0.12.1" \
    "jiwer" \
    "gTTS" \
    "edge-tts" \
    "datasets" \
    "huggingface_hub" \
    "torchao>=0.16.0"

## 2. Check GPUs and Storage

In [1]:
import torch
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("GPUs visible:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"cuda:{i} -> {torch.cuda.get_device_name(i)}")

GPUs visible: 1
cuda:0 -> Tesla T4


In [2]:
!nvidia-smi

Sat Aug 29 04:13:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [28]:
!df -h /kaggle/working 2>/dev/null || df -h .

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  272M   20G   2% /kaggle/working


## 3. Config

- `LLM_BASE` at 7B — best quality, still fast enough for a few hundred tokens of output once merged to fp16 (see Section 6).
- Switch `LLM_BASE` to `"Qwen/Qwen2.5-3B-Instruct"` — noticeably faster time-to-first-token and total generation time, small quality trade-off.

In [4]:
# Whisper: distil-whisper is purpose-built for low-latency ASR
WHISPER_BASE = "distil-whisper/distil-small.en"

# LLM: 7B for quality (swap to "Qwen/Qwen2.5-3B-Instruct" for lower latency)
LLM_BASE = "Qwen/Qwen2.5-7B-Instruct"

# Where trained adapters get saved
WHISPER_ADAPTER_DIR = "whisper-lora-out"
LLM_ADAPTER_DIR = "llm-qlora-out"

# Where merged, serving-ready models get saved
WHISPER_MERGED_DIR = "whisper-merged"
LLM_MERGED_DIR = "llm-merged"

# Data settings
MAX_EXAMPLES = 1500
VAL_FRACTION = 0.05
CONCURRENCY = 16

# Training settings
WHISPER_EPOCHS = 20
WHISPER_BATCH_SIZE = 16
LLM_EPOCHS = 5
LLM_BATCH_SIZE = 4

## 4. Data preparation

Public disfluency / grammar-correction **text** dataset for two different purposes:

- **LLM cleanup training**: the `(noisy, clean)` text pairs, used
  directly — this is where the actual cleanup transformation should live.
- **Whisper training**: TTS-synthesized audio of the **noisy** sentence,
  paired with a transcript label that is *also* the noisy sentence (not
  the clean one). Whisper's only job is to transcribe **what was said,
  accurately**

In [5]:
# (noisy_text, clean_text) pairs

pairs = []

# Load the Kaggle speech-cleanup-dataset
import kagglehub
from kagglehub import KaggleDatasetAdapter

DATASET_HANDLE = "bariankitvinod/speech-cleanup-dataset"
FILE_PATH = "speech_cleanup.parquet"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    DATASET_HANDLE,
    FILE_PATH,
)

print("Loaded dataset:", DATASET_HANDLE)
print("Columns:", df.columns.tolist())
print("Rows:", len(df))

# Automatically identify noisy/clean columns
NOISY_COL = "noisy_text"
CLEAN_COL = "clean_text"

if NOISY_COL not in df.columns or CLEAN_COL not in df.columns:
    raise ValueError(
        f"Expected columns '{NOISY_COL}' and '{CLEAN_COL}', "
        f"but found: {df.columns.tolist()}"
    )

# Build (noisy_text, clean_text) pairs
for _, row in df.iterrows():
    noisy = row[NOISY_COL]
    clean = row[CLEAN_COL]

    # Some datasets may store multiple reference corrections as a list
    if isinstance(clean, list):
        clean = clean[0] if clean else None

    if noisy is None or clean is None:
        continue

    noisy = str(noisy).strip()
    clean = str(clean).strip()

    if not noisy or not clean:
        continue

    if noisy == clean:
        continue  # no cleanup signal, skip

    word_count = len(noisy.split())

    if word_count < 3 or word_count > 40:
        continue  # keep clips short so TTS + training stays fast

    pairs.append((noisy, clean))

print("Total combined pairs:", len(pairs))

/tmp/ipykernel_257/3611181545.py:12: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Loaded dataset: bariankitvinod/speech-cleanup-dataset
Columns: ['noisy_text', 'clean_text', 'source']
Rows: 7935
Total combined pairs: 7786


In [6]:
import random

random.seed(42)
random.shuffle(pairs)
pairs = pairs[:MAX_EXAMPLES]

num_val = max(1, int(len(pairs) * VAL_FRACTION))
val_pairs = pairs[:num_val]
train_pairs = pairs[num_val:]

print("Train pairs:", len(train_pairs))
print("Val pairs:", len(val_pairs))

Train pairs: 1425
Val pairs: 75


In [7]:
import json
import os

os.makedirs("data", exist_ok=True)

def write_jsonl(path, split_pairs):
    with open(path, "w") as f:
        for noisy, clean in split_pairs:
            f.write(json.dumps({"noisy": noisy, "clean": clean}) + "\n")

write_jsonl("data/llm_train.jsonl", train_pairs)
write_jsonl("data/llm_val.jsonl", val_pairs)

In [8]:
import asyncio
import csv
import random

os.makedirs("data/audio", exist_ok=True)

import nest_asyncio
nest_asyncio.apply()

EDGE_VOICES = [
    "en-US-AriaNeural", "en-US-GuyNeural", "en-US-JennyNeural", "en-GB-SoniaNeural", "en-GB-RyanNeural", "en-AU-NatashaNeural", "en-IN-NeerjaNeural",
]

try:
    import edge_tts
    EDGE_TTS_AVAILABLE = True
except ImportError:
    EDGE_TTS_AVAILABLE = False
    print("edge-tts not installed/reachable -- falling back to gTTS (single voice).")

semaphore = asyncio.Semaphore(CONCURRENCY)

async def _synth_edge(text, voice, out_path):
    async with semaphore:
        communicate = edge_tts.Communicate(text, voice)
        await communicate.save(out_path)

def synthesize_one_gtts(text, out_path):
    from gtts import gTTS
    gTTS(text=text, lang="en").save(out_path)

async def synthesize_split_async(split_name, split_pairs):
    manifest_path = f"data/whisper_manifest_{split_name}.csv"
    rng = random.Random(42)

    tasks = []
    rows = []  # (audio_path, text, needs_gtts_fallback_flag placeholder)

    for i, (noisy, clean) in enumerate(split_pairs):
        audio_path = f"data/audio/{split_name}_{i:05d}.mp3"
        rows.append((audio_path, noisy))
        if not os.path.exists(audio_path) and EDGE_TTS_AVAILABLE:
            voice = rng.choice(EDGE_VOICES)
            tasks.append(_synth_edge(noisy, voice, audio_path))
        elif not os.path.exists(audio_path):
            synthesize_one_gtts(noisy, audio_path)

    if tasks:
        results = await asyncio.gather(*tasks, return_exceptions=True)
        for r in results:
            if isinstance(r, Exception):
                print("edge-tts task failed:", r)

    # gTTS fallback for any edge-tts tasks that failed and left no file
    for audio_path, noisy in rows:
        if not os.path.exists(audio_path):
            try:
                synthesize_one_gtts(noisy, audio_path)
            except Exception as error:
                print("gTTS fallback also failed for", audio_path, ":", error)

    with open(manifest_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["audio_path", "text"])
        for audio_path, noisy in rows:
            if os.path.exists(audio_path):
                writer.writerow([audio_path, noisy])

    print(f"[{split_name}] done: {sum(os.path.exists(p) for p, _ in rows)}/{len(rows)} synthesized")

asyncio.run(synthesize_split_async("train", train_pairs))
asyncio.run(synthesize_split_async("val", val_pairs))

[train] done: 1425/1425 synthesized
[val] done: 75/75 synthesized


## 5. Fine-tune Whisper (LoRA)

Fine-tune targets **robustness to messy/informal speech audio**, not content rewriting -- Whisper is trained to transcribe the noisy
sentence it hears **verbatim** (see the Section 4 note).

In [9]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

whisper_processor = WhisperProcessor.from_pretrained(WHISPER_BASE)

whisper_model = WhisperForConditionalGeneration.from_pretrained(
    WHISPER_BASE,
    attn_implementation="sdpa",
)

whisper_model.generation_config.forced_decoder_ids = None

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [10]:
from peft import LoraConfig, get_peft_model

whisper_lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

whisper_model = get_peft_model(
    whisper_model,
    whisper_lora_config
)

whisper_model.print_trainable_parameters()

trainable params: 1,966,080 || all params: 168,098,304 || trainable%: 1.1696


In [11]:
import librosa
import torch as torch_module  # avoid shadowing the torch import above

SAMPLING_RATE = 16000

class ManifestAudioDataset(torch_module.utils.data.Dataset):
    """Reads (audio_path, text) rows from a CSV and featurizes audio lazily."""

    def __init__(self, manifest_path, processor):
        self.rows = []
        with open(manifest_path) as f:
            reader = csv.DictReader(f)
            for row in reader:
                self.rows.append(row)
        self.processor = processor

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        audio, _ = librosa.load(row["audio_path"], sr=SAMPLING_RATE)

        input_features = self.processor.feature_extractor(
            audio, sampling_rate=SAMPLING_RATE
        ).input_features[0]

        labels = self.processor.tokenizer(row["text"]).input_ids

        return {"input_features": input_features, "labels": labels}

print("Dataset class ready.")

Dataset class ready.


In [12]:
class DataCollatorSpeechSeq2Seq:
    """Pads audio features and text labels separately for a training batch."""

    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels

        # expected by the model's encoder.
        model_dtype = next(whisper_model.parameters()).dtype
        batch["input_features"] = batch["input_features"].to(dtype=model_dtype)
        
        return batch

whisper_collator = DataCollatorSpeechSeq2Seq(whisper_processor)
print("Collator ready.")

Collator ready.


In [13]:
whisper_train_ds = ManifestAudioDataset("data/whisper_manifest_train.csv", whisper_processor)
whisper_val_ds = ManifestAudioDataset("data/whisper_manifest_val.csv", whisper_processor)

print("Whisper train examples:", len(whisper_train_ds))
print("Whisper val examples:", len(whisper_val_ds))

Whisper train examples: 1425
Whisper val examples: 75


In [14]:
import evaluate

wer_metric = evaluate.load("wer")

def compute_wer(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = whisper_processor.tokenizer.pad_token_id

    pred_str = whisper_processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = whisper_processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [15]:
from transformers import Seq2SeqTrainingArguments

num_update_steps_per_epoch = max(1, len(whisper_train_ds) // (WHISPER_BATCH_SIZE * 2))
total_train_steps = num_update_steps_per_epoch * WHISPER_EPOCHS
warmup_steps = int(0.1 * total_train_steps)

whisper_training_args = Seq2SeqTrainingArguments(
    output_dir=WHISPER_ADAPTER_DIR,
    per_device_train_batch_size=WHISPER_BATCH_SIZE,
    per_device_eval_batch_size=WHISPER_BATCH_SIZE,
    gradient_accumulation_steps=2,
    learning_rate=1e-3,
    warmup_steps=warmup_steps,
    num_train_epochs=WHISPER_EPOCHS,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=225,
    report_to="none",
    label_names=["labels"],
    remove_unused_columns=False,
)

In [16]:
from transformers import Seq2SeqTrainer

whisper_trainer = Seq2SeqTrainer(
    model=whisper_model,
    args=whisper_training_args,
    data_collator=whisper_collator,
    train_dataset=whisper_train_ds,
    eval_dataset=whisper_val_ds,
    processing_class=whisper_processor.feature_extractor,
    compute_metrics=compute_wer,
)

In [17]:
whisper_trainer.train()

You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Wer
1,1.452700,0.518260,11.886305
2,0.346200,0.472725,12.144703
3,0.231700,0.418568,10.938846
4,0.139600,0.424013,11.541774
5,0.110800,0.456888,10.938846
6,0.071300,0.445909,9.905254
7,0.061700,0.493033,10.766581
8,0.048400,0.488129,11.111111
9,0.029100,0.481870,9.905254
10,0.023400,0.507484,9.991387


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


TrainOutput(global_step=900, training_loss=0.10666028190818098, metrics={'train_runtime': 3066.1648, 'train_samples_per_second': 9.295, 'train_steps_per_second': 0.294, 'total_flos': 5.20266903552e+18, 'train_loss': 0.10666028190818098, 'epoch': 20.0})

In [18]:
whisper_model.save_pretrained(f"{WHISPER_ADAPTER_DIR}/final_adapter")
whisper_processor.save_pretrained(f"{WHISPER_ADAPTER_DIR}/final_adapter")

[]

## 6. Fine-tune the LLM (QLoRA)

Same text pairs, now training the cleanup LLM. The base model is loaded in **4-bit** (QLoRA) so a 7B model fits on a single T4 during training.

In [19]:
from transformers import AutoTokenizer

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_BASE)

if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [20]:
CLEANUP_SYSTEM_PROMPT = (
    "You are a transcription cleanup assistant. Rewrite the raw speech "
    "transcript into clear, well-punctuated text. Remove filler words and "
    "false starts, fix grammar. Do not add information that was not said. "
    "Output ONLY the cleaned text."
)

def build_chat_example(row):
    messages = [
        {"role": "system", "content": CLEANUP_SYSTEM_PROMPT},
        {"role": "user", "content": row["noisy"]},
        {"role": "assistant", "content": row["clean"]},
    ]
    text = llm_tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

llm_train_rows = [build_chat_example(r) for r in load_jsonl("data/llm_train.jsonl")]
llm_val_rows = [build_chat_example(r) for r in load_jsonl("data/llm_val.jsonl")]

print("LLM train examples:", len(llm_train_rows))
print("LLM val examples:", len(llm_val_rows))

LLM train examples: 1425
LLM val examples: 75


In [21]:
from datasets import Dataset

MAX_LENGTH = 512

def tokenize_fn(examples):
    return llm_tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

llm_train_ds = Dataset.from_list(llm_train_rows).map(tokenize_fn, batched=True, remove_columns=["text"])
llm_val_ds = Dataset.from_list(llm_val_rows).map(tokenize_fn, batched=True, remove_columns=["text"])

Map:   0%|          | 0/1425 [00:00<?, ? examples/s]

Map:   0%|          | 0/75 [00:00<?, ? examples/s]

In [22]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_BASE,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
)

llm_model = prepare_model_for_kbit_training(llm_model)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [23]:
llm_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

llm_model = get_peft_model(llm_model, llm_lora_config)
llm_model.print_trainable_parameters()

trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


In [24]:
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

llm_collator = DataCollatorForLanguageModeling(
    tokenizer=llm_tokenizer,
    mlm=False,
)

llm_training_args = TrainingArguments(
    output_dir=LLM_ADAPTER_DIR,
    num_train_epochs=LLM_EPOCHS,
    per_device_train_batch_size=LLM_BATCH_SIZE,
    per_device_eval_batch_size=LLM_BATCH_SIZE,
    gradient_accumulation_steps=4,
    optim="adamw_torch",
    learning_rate=2e-4,
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.999,
    adam_epsilon=1e-8,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    max_grad_norm=1.0,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    logging_strategy="steps",
    logging_steps=10,
    logging_first_step=True,
    report_to="none",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=42,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    remove_unused_columns=True,
)

In [25]:
llm_trainer = Trainer(
    model=llm_model,
    args=llm_training_args,
    train_dataset=llm_train_ds,
    eval_dataset=llm_val_ds,
    data_collator=llm_collator,
)

In [26]:
llm_trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss
1,0.756500,0.825600
2,0.729900,0.778667
3,0.649500,0.776241
4,0.636600,0.779703
5,0.605100,0.791095


TrainOutput(global_step=450, training_loss=0.8997471819983588, metrics={'train_runtime': 2787.9612, 'train_samples_per_second': 2.556, 'train_steps_per_second': 0.161, 'total_flos': 3.107503438147584e+16, 'train_loss': 0.8997471819983588, 'epoch': 5.0})

In [27]:
llm_model.save_pretrained(f"{LLM_ADAPTER_DIR}/final_adapter")
llm_tokenizer.save_pretrained(f"{LLM_ADAPTER_DIR}/final_adapter")

('llm-qlora-out/final_adapter/tokenizer_config.json',
 'llm-qlora-out/final_adapter/special_tokens_map.json',
 'llm-qlora-out/final_adapter/chat_template.jinja',
 'llm-qlora-out/final_adapter/vocab.json',
 'llm-qlora-out/final_adapter/merges.txt',
 'llm-qlora-out/final_adapter/added_tokens.json',
 'llm-qlora-out/final_adapter/tokenizer.json')

## 7. Custom CUDA kernel (training-speed optimization)

A hand-written kernel that fuses "add bias" + "GELU activation" into one pass, instead of two separate elementwise ops. Elementwise ops like this are memory-bandwidth bound, not compute bound — fusing them halves how many times each value is read/written from GPU memory, which is where the speedup actually comes from.

In [29]:
cuda_kernel_source = r'''
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <math.h>

#define SQRT_2_OVER_PI 0.7978845608028654f
#define GELU_COEF 0.044715f

__device__ __forceinline__ float gelu_fwd(float x) {
    float x3 = x * x * x;
    float inner = SQRT_2_OVER_PI * (x + GELU_COEF * x3);
    return 0.5f * x * (1.0f + tanhf(inner));
}

__device__ __forceinline__ float gelu_bwd(float x) {
    float x2 = x * x;
    float x3 = x2 * x;
    float inner = SQRT_2_OVER_PI * (x + GELU_COEF * x3);
    float tanh_inner = tanhf(inner);
    float sech2 = 1.0f - tanh_inner * tanh_inner;
    float d_inner = SQRT_2_OVER_PI * (1.0f + 3.0f * GELU_COEF * x2);
    return 0.5f * (1.0f + tanh_inner) + 0.5f * x * sech2 * d_inner;
}

__global__ void fused_bias_gelu_fwd_kernel(
    const float* __restrict__ x, const float* __restrict__ bias,
    float* __restrict__ y, int rows, int cols
) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    int total = rows * cols;
    for (int i = idx; i < total; i += stride) {
        int col = i % cols;
        float val = x[i] + bias[col];
        y[i] = gelu_fwd(val);
    }
}

__global__ void fused_bias_gelu_bwd_kernel(
    const float* __restrict__ grad_out, const float* __restrict__ x,
    const float* __restrict__ bias, float* __restrict__ grad_x,
    float* __restrict__ grad_bias, int rows, int cols
) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    int total = rows * cols;
    for (int i = idx; i < total; i += stride) {
        int col = i % cols;
        float val = x[i] + bias[col];
        float local_grad = gelu_bwd(val) * grad_out[i];
        grad_x[i] = local_grad;
        atomicAdd(&grad_bias[col], local_grad);
    }
}

torch::Tensor fused_bias_gelu_forward(torch::Tensor x, torch::Tensor bias) {
    x = x.contiguous();
    bias = bias.contiguous();
    int rows = x.size(0);
    int cols = x.size(1);
    auto y = torch::empty_like(x);
    int threads = 256;
    int total = rows * cols;
    int blocks = std::min((total + threads - 1) / threads, 65535);
    fused_bias_gelu_fwd_kernel<<<blocks, threads>>>(
        x.data_ptr<float>(), bias.data_ptr<float>(), y.data_ptr<float>(), rows, cols
    );
    return y;
}

std::vector<torch::Tensor> fused_bias_gelu_backward(
    torch::Tensor grad_out, torch::Tensor x, torch::Tensor bias
) {
    grad_out = grad_out.contiguous();
    x = x.contiguous();
    bias = bias.contiguous();
    int rows = x.size(0);
    int cols = x.size(1);
    auto grad_x = torch::empty_like(x);
    auto grad_bias = torch::zeros_like(bias);
    int threads = 256;
    int total = rows * cols;
    int blocks = std::min((total + threads - 1) / threads, 65535);
    fused_bias_gelu_bwd_kernel<<<blocks, threads>>>(
        grad_out.data_ptr<float>(), x.data_ptr<float>(), bias.data_ptr<float>(),
        grad_x.data_ptr<float>(), grad_bias.data_ptr<float>(), rows, cols
    );
    return {grad_x, grad_bias};
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &fused_bias_gelu_forward, "Fused bias+GELU forward (CUDA)");
    m.def("backward", &fused_bias_gelu_backward, "Fused bias+GELU backward (CUDA)");
}
'''

with open("fused_bias_gelu_kernel.cu", "w") as f:
    f.write(cuda_kernel_source)

In [30]:
from torch.utils.cpp_extension import load

fused_bias_gelu_ext = load(
    name="fused_bias_gelu_ext",
    sources=["fused_bias_gelu_kernel.cu"],
    verbose=True,
)

[1/2] /usr/local/cuda/bin/nvcc --generate-dependencies-with-compile --dependency-output fused_bias_gelu_kernel.cuda.o.d -DTORCH_EXTENSION_NAME=fused_bias_gelu_ext -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.12/dist-packages/torch/include -isystem /usr/local/lib/python3.12/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/cuda/include -isystem /usr/include/python3.12 -D__CUDA_NO_HALF_OPERATORS__ -D__CUDA_NO_HALF_CONVERSIONS__ -D__CUDA_NO_BFLOAT16_CONVERSIONS__ -D__CUDA_NO_HALF2_OPERATORS__ --expt-relaxed-constexpr -gencode=arch=compute_75,code=compute_75 -gencode=arch=compute_75,code=sm_75 --compiler-options '-fPIC' -std=c++17 -c /kaggle/working/fused_bias_gelu_kernel.cu -o fused_bias_gelu_kernel.cuda.o 
[2/2] c++ fused_bias_gelu_kernel.cuda.o -shared -L/usr/local/lib/python3.12/dist-packages/torch/lib -lc10 -lc10_cuda -ltorch_cpu -ltorch_cuda -ltorch -ltorch_python -L/usr/local/cuda/lib64 -lcudart -o fused_bias_gelu_ext.so


In [31]:
class FusedBiasGELU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bias):
        y = fused_bias_gelu_ext.forward(x, bias)
        ctx.save_for_backward(x, bias)
        return y

    @staticmethod
    def backward(ctx, grad_output):
        x, bias = ctx.saved_tensors
        grad_x, grad_bias = fused_bias_gelu_ext.backward(grad_output, x, bias)
        return grad_x, grad_bias

def fused_bias_gelu(x, bias):
    return FusedBiasGELU.apply(x, bias)

In [32]:
import torch.nn.functional as F

def torch_reference(x, bias):
    return F.gelu(x + bias, approximate="tanh")

torch.manual_seed(0)
x = torch.randn(64, 1024, device="cuda", requires_grad=True)
bias = torch.randn(1024, device="cuda", requires_grad=True)

x_ref = x.detach().clone().requires_grad_(True)
bias_ref = bias.detach().clone().requires_grad_(True)

y_custom = fused_bias_gelu(x, bias)
y_ref = torch_reference(x_ref, bias_ref)

forward_matches = torch.allclose(y_custom, y_ref, atol=1e-4, rtol=1e-4)
print("Forward output matches PyTorch:", forward_matches)

grad_out = torch.randn_like(y_custom)
y_custom.backward(grad_out)
y_ref.backward(grad_out)

grad_x_matches = torch.allclose(x.grad, x_ref.grad, atol=1e-4, rtol=1e-4)
grad_bias_matches = torch.allclose(bias.grad, bias_ref.grad, atol=1e-3, rtol=1e-3)
print("grad_x matches PyTorch:", grad_x_matches)
print("grad_bias matches PyTorch:", grad_bias_matches)

Forward output matches PyTorch: True
grad_x matches PyTorch: True
grad_bias matches PyTorch: True


In [33]:
import time

def benchmark_op(fn, x, bias, iters=50):
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(iters):
        x.grad = None
        bias.grad = None
        y = fn(x, bias)
        y.backward(torch.ones_like(y))
    torch.cuda.synchronize()
    return (time.time() - start) / iters * 1000  # milliseconds per iteration

x = torch.randn(4096, 4096, device="cuda", requires_grad=True)
bias = torch.randn(4096, device="cuda", requires_grad=True)

benchmark_op(fused_bias_gelu, x, bias, iters=5)
benchmark_op(torch_reference, x, bias, iters=5)

custom_ms = benchmark_op(fused_bias_gelu, x, bias)
reference_ms = benchmark_op(torch_reference, x, bias)

print(f"Custom fused kernel: {custom_ms:.3f} ms/iter")
print(f"PyTorch (add + gelu): {reference_ms:.3f} ms/iter")
print(f"Speedup: {reference_ms / custom_ms:.2f}x")

Custom fused kernel: 2.121 ms/iter
PyTorch (add + gelu): 2.507 ms/iter
Speedup: 1.18x


## 8. Merge adapters for low-latency serving

LoRA adapters add a small amount of compute overhead at inference (an extra matmul per adapted layer). **Merging** folds the adapter weights directly into the base model, so serving runs at the same speed as the original base model — this matters for a low-latency target.

In [34]:
from peft import PeftModel

whisper_base_for_merge = WhisperForConditionalGeneration.from_pretrained(WHISPER_BASE)
whisper_merged = PeftModel.from_pretrained(whisper_base_for_merge, f"{WHISPER_ADAPTER_DIR}/final_adapter")
whisper_merged = whisper_merged.merge_and_unload()

os.makedirs(WHISPER_MERGED_DIR, exist_ok=True)
whisper_merged.save_pretrained(WHISPER_MERGED_DIR)
whisper_processor.save_pretrained(WHISPER_MERGED_DIR)

/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 357, 366, 438, 532, 685, 705, 796, 930, 1058, 1220, 1267, 1279, 1303, 1343, 1377, 1391, 1635, 1782, 1875, 2162, 2361, 2488, 3467, 4008, 4211, 4600, 4808, 5299, 5855, 6329, 7203, 9609, 9959, 10563, 10786, 11420, 11709, 11907, 13163, 13697, 13700, 14808, 15306, 16410, 16791, 17992, 19203, 19510, 20724, 22305, 22935, 27007, 30109, 30420, 33409, 34949, 40283, 40493, 40549, 47282, 49146, 50257, 50357, 50358, 50359, 50360, 50361]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


[]

In [35]:
llm_base_fp16 = AutoModelForCausalLM.from_pretrained(
    LLM_BASE,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
)

llm_merged = PeftModel.from_pretrained(llm_base_fp16, f"{LLM_ADAPTER_DIR}/final_adapter")
llm_merged = llm_merged.merge_and_unload()

os.makedirs(LLM_MERGED_DIR, exist_ok=True)
llm_merged.save_pretrained(LLM_MERGED_DIR)
llm_tokenizer.save_pretrained(LLM_MERGED_DIR)

print("Saved merged LLM to", LLM_MERGED_DIR)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1566: UserWarning: Current model requires 256 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3970: UserWarning: Attempting to save a model with offloaded modules. Ensure that unallocated cpu memory exceeds the `shard_size` (5GB default)
  warnings.warn(


Saving checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Saved merged LLM to llm-merged


## 9. End-to-end latency check

A quick sanity check that the merged, serving-ready pipeline is actually fast: transcribe a short audio clip, then generate the cleaned text, and time both stages.


In [36]:
from transformers import pipeline, TextIteratorStreamer

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=whisper_merged,
    tokenizer=whisper_processor.tokenizer,
    feature_extractor=whisper_processor.feature_extractor,
    torch_dtype=torch.float16,
    device="cuda:0",
)

test_audio_path = val_pairs and f"data/audio/val_00000.mp3"

start = time.time()
asr_result = asr_pipe(test_audio_path)
asr_seconds = time.time() - start

print("Raw transcript:", asr_result["text"])
print(f"ASR time: {asr_seconds * 1000:.0f} ms")

`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0
`return_token_timestamps` is deprecated for WhisperFeatureExtractor and will be removed in Transformers v5. Use `return_attention_mask` instead, as the number of frames can be inferred from it.


Raw transcript: What advancements besides explosives, no I mean military technology did Europe not achieve?
ASR time: 2779 ms


In [37]:
messages = [
    {"role": "system", "content": CLEANUP_SYSTEM_PROMPT},
    {"role": "user", "content": asr_result["text"]},
]

prompt = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = llm_tokenizer(prompt, return_tensors="pt").to(llm_merged.device)

start = time.time()
output_ids = llm_merged.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=False,
    repetition_penalty=1.1,
)
llm_seconds = time.time() - start

new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
cleaned_text = llm_tokenizer.decode(new_tokens, skip_special_tokens=True)

print("Cleaned text:", cleaned_text)
print(f"LLM time: {llm_seconds * 1000:.0f} ms  ({len(new_tokens) / llm_seconds:.1f} tokens/sec)")
print(f"Total end-to-end: {(asr_seconds + llm_seconds) * 1000:.0f} ms")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Cleaned text: What advancements besides military technology did Europe not achieve?
LLM time: 86530 ms  (0.1 tokens/sec)
Total end-to-end: 89309 ms


## 10. Push the merged models to Hugging Face Hub

In [ ]:
from huggingface_hub import login
login(token="hf_xxx")

In [39]:
HF_USERNAME = "aijadugar"
WHISPER_REPO_ID = f"{HF_USERNAME}/wisprflow-clone-whisper"
LLM_REPO_ID = f"{HF_USERNAME}/wisprflow-clone-llm"

whisper_merged.push_to_hub(WHISPER_REPO_ID)
whisper_processor.push_to_hub(WHISPER_REPO_ID)

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/aijadugar/wisprflow-clone-whisper/commit/017f126de3a6236dd34db19b07299ba53018ff89', commit_message='Upload processor', commit_description='', oid='017f126de3a6236dd34db19b07299ba53018ff89', pr_url=None, repo_url=RepoUrl('https://huggingface.co/aijadugar/wisprflow-clone-whisper', endpoint='https://huggingface.co', repo_type='model', repo_id='aijadugar/wisprflow-clone-whisper'), pr_revision=None, pr_num=None)

In [40]:
llm_merged.push_to_hub(LLM_REPO_ID)
llm_tokenizer.push_to_hub(LLM_REPO_ID)

README.md: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3970: UserWarning: Attempting to save a model with offloaded modules. Ensure that unallocated cpu memory exceeds the `shard_size` (5GB default)
  warnings.warn(


Saving checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/aijadugar/wisprflow-clone-llm/commit/8ecb5cff61f0507609dcdf8a488dd7b157a6f236', commit_message='Upload tokenizer', commit_description='', oid='8ecb5cff61f0507609dcdf8a488dd7b157a6f236', pr_url=None, repo_url=RepoUrl('https://huggingface.co/aijadugar/wisprflow-clone-llm', endpoint='https://huggingface.co', repo_type='model', repo_id='aijadugar/wisprflow-clone-llm'), pr_revision=None, pr_num=None)

## 11. Free local disk

In [ ]:
import shutil

for path in [WHISPER_MERGED_DIR, LLM_MERGED_DIR, WHISPER_ADAPTER_DIR, LLM_ADAPTER_DIR, "data/audio"]:
    if os.path.exists(path):
        shutil.rmtree(path)
        print("Removed", path)

!df -h /kaggle/working 2>/dev/null || df -h .